In [94]:
import pandas as pd
import numpy as np
import sys, csv, traceback
from pathlib import Path
from unidecode import unidecode
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)
src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from routes.Paths import Ruta_opl, Ruta_opl_historico

In [95]:
required_columns = [
        "Identificacion", "ID NICE", "ID Workday", "Nombre", "Jefe Inmediato", "Lider",
        "Posicion", "Servicio", "Servicio factura", "Unidad de negocio",
        "Genero", "Fecha de nacimiento", "Fecha Inicio Servicio", "Fecha de contratación",
        "Condición", "Grupo/Wave", "Site", "Estado", "HOJA", "Vista 360","Correo Corporativo"
    ]

In [96]:
df_opl = pd.read_excel(
    Ruta_opl,
     sheet_name="Combinado",
     engine="openpyxl")

df_opl = df_opl[required_columns]

for col in required_columns:
    df_opl[col] = (
        df_opl[col]
        .fillna("-")
        .astype(str)
        .str.strip()
        .map(unidecode)
    )

df_opl.columns = (df_opl.columns
                  .str.strip()
                  .str.upper()
                  .map(unidecode))

df_opl = df_opl[df_opl["ESTADO"].isin(["ACTIVO", "APOYO", "MATERNA"])]

columns_dates = [
        "FECHA DE NACIMIENTO", 
        "FECHA INICIO SERVICIO", 
        "FECHA DE CONTRATACION"
    ]
for col in columns_dates:
        df_opl[col] = pd.to_datetime(df_opl[col], format='%d/%m/%Y', errors='coerce')
df_opl = df_opl.drop_duplicates(["IDENTIFICACION", "ID WORKDAY", "VISTA 360"])


df_opl = df_opl.rename(columns={
  "VISTA 360":"CORREO CONTRATISTA"    
})

In [98]:
df_opl_historico = pd.read_excel(Ruta_opl_historico)
df_opl_historico = df_opl_historico.drop_duplicates(["IDENTIFICACION", "ID WORKDAY", "CORREO CONTRATISTA", "FECHA CARGA"],keep="last")